# MLOps Assignment 2: LLMOps - API Cost, Latency, and Prompt Version Management

**Course:** MLOps  
**Mode:** Individual hands-on activity  
**Marks:** 10  
**Author:** Gauri Shanker (2025AF05069)  
**Date:** 2026-11-01

## Objective

This assignment operationalizes an LLM application by:
1. Measuring latency for API calls
2. Calculating token-based costs
3. Managing and evaluating prompt versions in MLflow

## Learning Outcomes

After completing this assignment, you will be able to:
- Explain LLMOps and its differences from MLOps
- Call an LLM through an API and measure response latency
- Extract token usage and compute request-level and projected monthly costs
- Register, compare, evaluate, and promote prompt versions in MLflow
- Create a JSON-output prompt and validate it using a valid_json metric

## Setup and Imports

In [1]:
# Import standard libraries
import os
import json
import sys
import pandas as pd
import numpy as np
from datetime import datetime
import time
from typing import Dict, List

# Import project modules (works both from notebook root and from src package)
from src.config import (
    OPENAI_API_KEY,
    MODEL_NAME,
    ACADEMIC_QUESTIONS,
    PROMPTS,
    MLFLOW_EXPERIMENT_NAME,
    LATENCY_TOKENS_COST_CSV,
    PROMPT_EVALUATION_CSV
)
from src.latency import LLMAPIClient, LatencyAnalyzer, LatencyRecord
from src.cost import TokenCounter, CostCalculator, CostOptimizationAnalyzer
from src.mlflow_manager import (
    PromptRegistry,
    PromptEvaluator,
    MLflowExperimentTracker,
    PromptComparator
)

print("✓ All imports successful")

✓ All imports successful


## Task 1: API Call and Latency Measurement

**Objective:** Call an LLM API for five academic questions and measure latency.

**Requirements:**
- Load API key from environment variable
- Record question ID, prompt version, model name, start time, end time, latency, and response status
- Calculate mean, median, minimum, maximum, and standard deviation of latency
- Explain the slowest request and possible reasons

In [2]:
# Initialize API client
api_client = LLMAPIClient()

print(f"Initialized LLMAPIClient for {MODEL_NAME}")
print(f"API Key loaded: {bool(OPENAI_API_KEY)}")

Initialized LLMAPIClient for gpt-3.5-turbo
API Key loaded: True


In [3]:
# Call API for all questions with v1_basic prompt
print("\n" + "="*60)
print("Testing Latency with v1_basic Prompt")
print("="*60)

latency_records = api_client.test_latency_for_all_questions("v1_basic")

print(f"\n✓ Completed {len(latency_records)} API calls")


Testing Latency with v1_basic Prompt

✓ Completed 5 API calls


In [4]:
# Create latency summary table
latency_data = []

for record in latency_records:
    latency_data.append({
        'Question ID': record.question_id,
        'Prompt Version': record.prompt_version,
        'Model': record.model_name,
        'Start Time': record.start_time,
        'End Time': record.end_time,
        'Latency (seconds)': round(record.latency_seconds, 3),
        'Status': record.response_status,
        'Timestamp': record.timestamp
    })

latency_df = pd.DataFrame(latency_data)
print("\nLatency Table:")
print(latency_df.to_string())

# Save to CSV
latency_df.to_csv(LATENCY_TOKENS_COST_CSV, index=False)
print(f"\n✓ Latency data saved to {LATENCY_TOKENS_COST_CSV}")


Latency Table:
  Question ID Prompt Version          Model    Start Time      End Time  Latency (seconds) Status                   Timestamp
0          q1       v1_basic  gpt-3.5-turbo  1.793519e+09  1.793519e+09              0.659  error  2026-11-01T13:20:50.937592
1          q2       v1_basic  gpt-3.5-turbo  1.793519e+09  1.793519e+09              0.565  error  2026-11-01T13:20:52.100603
2          q3       v1_basic  gpt-3.5-turbo  1.793519e+09  1.793519e+09              0.280  error  2026-11-01T13:20:53.170571
3          q4       v1_basic  gpt-3.5-turbo  1.793519e+09  1.793519e+09              0.378  error  2026-11-01T13:20:53.955284
4          q5       v1_basic  gpt-3.5-turbo  1.793519e+09  1.793519e+09              0.353  error  2026-11-01T13:20:54.833630

✓ Latency data saved to outputs/latency_tokens_cost.csv


In [5]:
# Calculate latency statistics
analyzer = LatencyAnalyzer()
analysis = analyzer.analyze_records(latency_records)

print("\nLatency Statistics:")
print("-" * 40)
stats = analysis['latency_stats']
print(f"Total Requests: {analysis['total_requests']}")
print(f"Successful: {analysis['successful_requests']}")
print(f"Failed: {analysis['failed_requests']}")

# Handle case where no successful requests
if stats.get('count', 0) == 0:
    print("\n⚠ No successful API calls - stats are zero/default")
    print(f"Note: {stats.get('note', 'Check API key and network connectivity')}")
else:
    print(f"Mean Latency: {stats['mean_seconds']:.3f} seconds")
    print(f"Median Latency: {stats['median_seconds']:.3f} seconds")
    print(f"Min Latency: {stats['min_seconds']:.3f} seconds")
    print(f"Max Latency: {stats['max_seconds']:.3f} seconds")
    print(f"Std Dev: {stats['stdev_seconds']:.3f} seconds")
    
    if 'slowest_request' in analysis:
        print(f"\nSlowest Request:")
        print(f"  Question: {analysis['slowest_request']['question']}")
        print(f"  Latency: {analysis['slowest_request']['latency_seconds']:.3f} seconds")



Latency Statistics:
----------------------------------------
Total Requests: 5
Successful: 0
Failed: 5

⚠ No successful API calls - stats are zero/default
Note: No successful API calls available for statistics


## Task 2: Token Counting and Cost Calculation

**Objective:** Extract token usage and calculate costs based on API pricing.

**Formulas:**
- Input Cost = Input Tokens / 1,000,000 × Input Price per Million Tokens
- Output Cost = Output Tokens / 1,000,000 × Output Price per Million Tokens
- Total Cost per Request = Input Cost + Output Cost
- Projected Monthly Cost = Average Cost per Request × 10,000

In [6]:
# Extract token usage from responses and calculate costs
print("\n" + "="*60)
print("Token Counting and Cost Analysis")
print("="*60)

token_cost_data = []
total_costs = []

for record in latency_records:
    if record.response_status == 'success' and record.response_text:
        # Estimate tokens (in real scenario, extract from API response)
        # Note: OpenAI API includes token usage in response.usage
        question = next((q['question'] for q in ACADEMIC_QUESTIONS if q['id'] == record.question_id), '')
        formatted_prompt = PROMPTS['v1_basic']['template'].replace('{{question}}', question)
        
        input_tokens = len(formatted_prompt.split()) * 1.3  # Rough estimate
        output_tokens = len(record.response_text.split()) * 1.3
        
        # Calculate cost
        cost_calc = CostCalculator.calculate_cost_for_tokens(
            int(input_tokens),
            int(output_tokens),
            MODEL_NAME
        )
        
        total_costs.append(cost_calc.total_cost_per_request)
        
        token_cost_data.append({
            'Question ID': record.question_id,
            'Input Tokens': int(input_tokens),
            'Output Tokens': int(output_tokens),
            'Total Tokens': int(input_tokens + output_tokens),
            'Input Cost ($)': f"{cost_calc.input_cost:.6f}",
            'Output Cost ($)': f"{cost_calc.output_cost:.6f}",
            'Total Cost ($)': f"{cost_calc.total_cost_per_request:.6f}"
        })

token_cost_df = pd.DataFrame(token_cost_data)
print("\nToken and Cost Table:")
print(token_cost_df.to_string())


Token Counting and Cost Analysis

Token and Cost Table:
Empty DataFrame
Columns: []
Index: []


In [7]:
# Calculate cost summary and monthly projection
if total_costs:
    avg_cost = sum(total_costs) / len(total_costs)
    monthly_cost = CostCalculator.project_monthly_cost(avg_cost, 10_000)
    
    print("\nCost Summary:")
    print("-" * 40)
    print(f"Average Cost per Request: ${avg_cost:.6f}")
    print(f"Minimum Cost: ${min(total_costs):.6f}")
    print(f"Maximum Cost: ${max(total_costs):.6f}")
    print(f"\nProjected Monthly Cost (10,000 requests): ${monthly_cost:.2f}")
    print(f"Projected Annual Cost: ${monthly_cost * 12:.2f}")

In [8]:
# Cost optimization recommendations
print("\n" + "="*60)
print("Cost Optimization Recommendations")
print("="*60)

optimizer = CostOptimizationAnalyzer()
token_usage_list = [{
    'input_tokens': int(row['Input Tokens']),
    'output_tokens': int(row['Output Tokens'])
} for _, row in token_cost_df.iterrows()]

recommendations = optimizer.recommend_optimizations(token_usage_list, [])
for rec in recommendations:
    print(f"\n• {rec}")


Cost Optimization Recommendations

• 3. Implement Prompt Caching: For repeated queries with similar context, implement prompt caching at the API level to avoid re-processing stable system prompts and retrieved context (if using gpt-4-turbo or similar).

• 4. Model Optimization: Consider using gpt-3.5-turbo for simpler queries and evaluate gpt-4 only for complex tasks. For high-volume operations, evaluate cost vs. quality tradeoffs.

• 5. Batch Processing: If applicable, batch multiple queries into a single API call to reduce overhead and API invocation costs.


## Task 3: MLflow Prompt Versioning and Evaluation

**Objective:** Register and evaluate three prompt versions in MLflow.

**Prompts:**
- v1: Basic
- v2: Structured
- v3: JSON Output

**Metrics:**
- avg_latency_seconds
- avg_input_tokens
- avg_output_tokens
- avg_cost_per_request
- valid_json (for v3)

In [9]:
# Initialize MLflow components
print("\n" + "="*60)
print("Setting up MLflow")
print("="*60)

registry = PromptRegistry()
print(f"✓ Experiment created: {MLFLOW_EXPERIMENT_NAME}")
print(f"  Experiment ID: {registry.experiment_id}")

# Register all prompts
registered = registry.register_all_prompts()
print(f"\n✓ Registered {len(registered)} prompt versions")
for version, info in registered.items():
    print(f"  {version}: {info['name']}")


Setting up MLflow
✓ Experiment created: eduassist_llmops_assignment
  Experiment ID: 1

✓ Registered 3 prompt versions
  v1: Basic
  v2: Structured
  v3: JSON Output


In [10]:
# Evaluate all prompt versions
print("\n" + "="*60)
print("Evaluating Prompt Versions")
print("="*60)

evaluator = PromptEvaluator(api_client)
evaluation_results = evaluator.evaluate_all_versions(PROMPTS)

print(f"\n✓ Evaluated {len(evaluation_results)} prompt versions")


Evaluating Prompt Versions

Evaluating Prompt Version: v1
  q1: ✓ Latency: 0.679s
  q2: ✓ Latency: 0.403s
  q3: ✓ Latency: 0.323s
  q4: ✓ Latency: 0.334s
  q5: ✓ Latency: 0.411s

Evaluating Prompt Version: v2
  q1: ✓ Latency: 0.377s
  q2: ✓ Latency: 0.355s
  q3: ✓ Latency: 0.336s
  q4: ✓ Latency: 0.360s
  q5: ✓ Latency: 0.365s

Evaluating Prompt Version: v3
  q1: ✓ Latency: 0.326s
  q2: ✓ Latency: 0.340s
  q3: ✓ Latency: 0.374s
  q4: ✓ Latency: 0.325s
  q5: ✓ Latency: 0.344s

✓ Evaluated 3 prompt versions


In [11]:
# Log runs in MLflow
print("\n" + "="*60)
print("Logging Runs to MLflow")
print("="*60)

tracker = MLflowExperimentTracker(registry)
run_ids = tracker.log_all_runs(evaluator)

print(f"\n✓ Logged {len(run_ids)} runs to MLflow")


Logging Runs to MLflow
✓ Logged run for v1: 85c33ed277224cf6b3fc3a72a2d6edb2
✓ Logged run for v2: 17f08658113c4477b59df1c5dd815315
✓ Logged run for v3: 261c07c9eba44d469385810bb1e3a0b1

✓ Logged 3 runs to MLflow


In [12]:
# Compare prompt versions
print("\n" + "="*60)
print("Comparing Prompt Versions")
print("="*60)

comparator = PromptComparator()
comparison = comparator.compare_versions(evaluator)

# Create comparison table
comparison_data = []
for version, metrics in comparison['versions'].items():
    comparison_data.append({
        'Prompt Version': version,
        'Avg Latency (s)': f"{metrics['avg_latency_seconds']:.3f}",
        'Avg Input Tokens': f"{metrics['avg_input_tokens']:.0f}",
        'Avg Output Tokens': f"{metrics['avg_output_tokens']:.0f}",
        'Avg Cost ($)': f"{metrics['avg_cost_per_request']:.6f}",
        'Valid JSON Rate': f"{metrics['valid_json_rate']:.2%}",
        'Recommended': '✓ YES' if version == comparison['best_version'] else 'NO'
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nPrompt Comparison Table:")
print(comparison_df.to_string())

# Save comparison results
comparison_df.to_csv(PROMPT_EVALUATION_CSV, index=False)
print(f"\n✓ Comparison saved to {PROMPT_EVALUATION_CSV}")


Comparing Prompt Versions

Prompt Comparison Table:
  Prompt Version Avg Latency (s) Avg Input Tokens Avg Output Tokens Avg Cost ($) Valid JSON Rate Recommended
0             v1           0.430               17                 0     0.000009           0.00%       ✓ YES
1             v2           0.359               35                 0     0.000018           0.00%          NO
2             v3           0.342               52                 0     0.000026           0.00%          NO

✓ Comparison saved to outputs/prompt_evaluation_results.csv


In [13]:
# Promote best version to production
best_version = comparison['best_version']
tracker.promote_version_to_production(best_version)

print(f"\n✓ {best_version} promoted to production")


PROMOTION TO PRODUCTION: v1
Version: v1
Promoted at: 2026-11-01T13:22:14.155504
Status: production
Alias: production

✓ v1 promoted to production


## Summary and Conclusions

### Key Findings

1. **Latency Analysis:**
   - Mean latency across all requests
   - Variation in response times based on prompt complexity

2. **Cost Analysis:**
   - Token usage patterns and costs per request
   - Projected monthly costs for scaled deployments

3. **Prompt Evaluation:**
   - Performance comparison across v1, v2, and v3
   - JSON validity metrics for structured outputs
   - Best version selection based on quality, latency, and cost

### Recommendations

Based on the analysis, the following recommendations are made for production deployment.

In [14]:
# Display MLflow tracking URI
print("\nMLflow Experiment Tracking:")
print("-" * 40)
print(f"Experiment Name: {MLFLOW_EXPERIMENT_NAME}")
print(f"Experiment ID: {registry.experiment_id}")
print(f"\nTo view results, run:")
print(f"  mlflow ui --port 5000")
print(f"\nThen navigate to:")
print(f"  http://localhost:5000")


MLflow Experiment Tracking:
----------------------------------------
Experiment Name: eduassist_llmops_assignment
Experiment ID: 1

To view results, run:
  mlflow ui --port 5000

Then navigate to:
  http://localhost:5000
